<a href="https://colab.research.google.com/github/Maoyuu0396/-/blob/main/zhengliu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 检查GPU是否可用
import torch
print(f"GPU可用: {torch.cuda.is_available()}")
print(f"GPU型号: {torch.cuda.get_device_name(0)}")
print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 挂载Google Drive（用于保存模型）
from google.colab import drive
drive.mount('/content/drive')

GPU可用: True
GPU型号: Tesla T4
显存: 15.64 GB


MessageError: Error: credential propagation was unsuccessful

In [ ]:
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS

# 安装依赖（Colab预装了PyTorch，只需安装项目依赖）
!pip install -e .
!pip install tensorboard  # 用于可视化训练

Cloning into 'F5-TTS'...
remote: Enumerating objects: 3949, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 3949 (delta 177), reused 109 (delta 108), pack-reused 3698 (from 4)
Receiving objects: 100% (3949/3949), 2.38 MiB | 7.16 MiB/s, done.
Resolving deltas: 100% (2379/2379), done.
/content/F5-TTS
Obtaining file:///content/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 80.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 20.9 MB/s e

In [ ]:
import os
if os.path.exists("/content/teacher_model"):
    print("✅ 教师模型已存在")
    print("文件列表：", os.listdir("/content/teacher_model")[:5])
else:
    print("❌ 教师模型不存在，需要下载")

❌ 教师模型不存在，需要下载


In [ ]:
#挂载google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 在Drive中创建工作目录
import os
drive_path = "/content/drive/MyDrive/f5tts_project"
os.makedirs(f"{drive_path}/models", exist_ok=True)
os.makedirs(f"{drive_path}/output", exist_ok=True)
os.makedirs(f"{drive_path}/cache", exist_ok=True)

print(f"✅ Drive工作目录创建成功：{drive_path}")
print("文件夹结构：")
print(f"  - 模型保存：{drive_path}/models")
print(f"  - 输出保存：{drive_path}/output")
print(f"  - 缓存保存：{drive_path}/cache")

✅ Drive工作目录创建成功：/content/drive/MyDrive/f5tts_project
文件夹结构：
  - 模型保存：/content/drive/MyDrive/f5tts_project/models
  - 输出保存：/content/drive/MyDrive/f5tts_project/output
  - 缓存保存：/content/drive/MyDrive/f5tts_project/cache


In [ ]:
# ==================== 第三步：下载教师模型到Drive ====================
from huggingface_hub import snapshot_download

# 设置Drive中的模型保存路径
teacher_model_path = f"{drive_path}/models/F5TTS-Base"

print(f"开始下载教师模型到：{teacher_model_path}")
print("（约2GB，需要几分钟...）")

snapshot_download(
    repo_id="SWivid/F5-TTS-Base",
    local_dir=teacher_model_path,
    local_dir_use_symlinks=False
)

print(f"✅ 教师模型下载完成！")
print(f"模型位置：{teacher_model_path}")

开始下载教师模型到：/content/drive/MyDrive/f5tts_project/models/F5TTS-Base
（约2GB，需要几分钟...）


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69b26654-6af06d8263b7edd54da92f30;11587548-be31-4069-b4e2-5a73d4da4a2d)

Repository Not Found for url: https://huggingface.co/api/models/SWivid/F5-TTS-Base/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
#login huggingface
from huggingface_hub import login

# 运行后会提示输入token，粘贴你刚才复制的即可
login()

In [ ]:
from huggingface_hub import snapshot_download

teacher_model_path = f"{drive_path}/models/F5TTS-Base"

print(f"开始下载教师模型到：{teacher_model_path}")
print("（约2GB，需要几分钟...）")

snapshot_download(
    repo_id="SWivid/F5-TTS",
    local_dir=teacher_model_path,
    token=True  # 这会使用刚刚login时保存的token
)

print(f"✅ 教师模型下载完成！")

开始下载教师模型到：/content/drive/MyDrive/f5tts_project/models/F5TTS-Base
（约2GB，需要几分钟...）


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ 教师模型下载完成！


In [ ]:
# 验证下载
model_files = os.listdir(teacher_model_path)
print(f"文件数量：{len(model_files)}")
print(f"前5个文件：{model_files[:5]}")

文件数量：7
前5个文件：['.cache', 'F5TTS_Base', 'F5TTS_Base_bigvgan', 'F5TTS_v1_Base', 'F5TTS_v1_Base_no_zero_init']


In [ ]:
# 找到教师模型的真实路径
import os

search_path = "/content/drive/MyDrive/f5tts_project/models"
print(f"搜索路径: {search_path}")

for root, dirs, files in os.walk(search_path):
    for file in files:
        if file.endswith('.safetensors'):
            print(f"\n✅ 找到模型文件: {os.path.join(root, file)}")
            print(f"模型所在目录: {root}")
            break  # 找到一个就行

搜索路径: /content/drive/MyDrive/f5tts_project/models

✅ 找到模型文件: /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.safetensors
模型所在目录: /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base

✅ 找到模型文件: /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_v1_Base/model_1250000.safetensors
模型所在目录: /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_v1_Base

✅ 找到模型文件: /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_v1_Base_no_zero_init/model_1250000.safetensors
模型所在目录: /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_v1_Base_no_zero_init


In [ ]:
!cat /content/F5-TTS/src/f5_tts/configs/F5TTS_Small_Distill.yaml

cat: /content/F5-TTS/src/f5_tts/configs/F5TTS_Small_Distill.yaml: No such file or directory


In [ ]:
# ==================== 创建蒸馏配置文件 ====================
import os

# 读取原有的Small配置作为基础
small_config_path = "/content/F5-TTS/src/f5_tts/configs/F5TTS_Small.yaml"
with open(small_config_path, 'r') as f:
    small_config = f.read()

# 添加蒸馏参数
distill_config = small_config + """
# ===== 蒸馏参数（新增）=====
distillation:
  teacher_model_path: "/content/drive/MyDrive/f5tts_project/models/F5TTs-Base/F5TTs_Base"
  temperature: 4.0
  alpha: 0.7

# 输出到Drive
hydra:
  run:
    dir: /content/drive/MyDrive/f5tts_project/output/distill_${now:%Y-%m-%d_%H-%M-%S}
"""

# 保存新文件
new_config_path = "/content/F5-TTS/src/f5_tts/configs/F5TTS_Small_Distill.yaml"
with open(new_config_path, 'w') as f:
    f.write(distill_config)

print(f"✅ 已创建蒸馏配置文件：{new_config_path}")
print(f"基于：{small_config_path}")

✅ 已创建蒸馏配置文件：/content/F5-TTS/src/f5_tts/configs/F5TTS_Small_Distill.yaml
基于：/content/F5-TTS/src/f5_tts/configs/F5TTS_Small.yaml


In [ ]:
# ==================== 打印完整的模型文件结构 ====================
import os

# 搜索根路径
search_path = "/content/drive/MyDrive/f5tts_project/models"
print("="*60)
print(f"🔍 搜索路径: {search_path}")
print("="*60)

# 递归遍历所有文件和文件夹
for root, dirs, files in os.walk(search_path):
    # 计算当前层级（用于缩进）
    level = root.replace(search_path, '').count(os.sep)
    indent = '  ' * level

    # 打印当前文件夹
    folder_name = os.path.basename(root)
    if level == 0:
        print(f"\n📁 {folder_name}/")
    else:
        print(f"\n{indent}📁 {folder_name}/")

    # 打印文件夹内的文件
    for file in sorted(files):
        if file.endswith('.safetensors'):
            # 模型文件用✅标记
            file_size = os.path.getsize(os.path.join(root, file)) / 1024 / 1024
            print(f"{indent}  ✅ {file} ({file_size:.2f} MB)")
        elif file.endswith('.yaml') or file.endswith('.json'):
            # 配置文件用📄标记
            print(f"{indent}  📄 {file}")
        else:
            # 其他文件
            print(f"{indent}    {file}")

print("\n" + "="*60)
print("✅ 打印完成！")

🔍 搜索路径: /content/drive/MyDrive/f5tts_project/models

📁 models/

  📁 F5TTS-Base/
      .gitattributes
      README.md

    📁 .cache/

      📁 huggingface/
          .gitignore

        📁 download/
            .gitattributes.metadata
            README.md.metadata

          📁 F5TTS_Base/
              model_1200000.pt.metadata
              model_1200000.safetensors.metadata
              vocab.txt.metadata

          📁 F5TTS_Base_bigvgan/
              model_1250000.pt.metadata

          📁 F5TTS_v1_Base/
              model_1250000.safetensors.metadata
              vocab.txt.metadata

          📁 F5TTS_v1_Base_no_zero_init/
              model_1250000.safetensors.metadata

    📁 F5TTS_Base/
        model_1200000.pt
      ✅ model_1200000.safetensors (1286.17 MB)
        vocab.txt

    📁 F5TTS_Base_bigvgan/
        model_1250000.pt

    📁 F5TTS_v1_Base/
      ✅ model_1250000.safetensors (1285.97 MB)
        vocab.txt

    📁 F5TTS_v1_Base_no_zero_init/
      ✅ model_1250000.safetensors 

In [ ]:
# 验证教师模型文件是否存在
import os

teacher_file = "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.safetensors"
if os.path.exists(teacher_file):
    print(f"✅ 教师模型文件存在")
    print(f"路径：{teacher_file}")
    print(f"大小：{os.path.getsize(teacher_file) / 1024 / 1024:.2f} MB")
else:
    print("❌ 文件不存在，请检查路径")

✅ 教师模型文件存在
路径：/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.safetensors
大小：1286.17 MB


In [ ]:
# 进入项目目录
%cd /content/F5-TTS

# 蒸馏测试命令（使用 .pt 文件）
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name ljspeech \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer pinyin \
    --log_samples

/content/F5-TTS
copy checkpoint for finetune
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/../../data/ljspeech_pinyin/vocab.txt'


In [ ]:
# 进入项目目录
%cd /content/F5-TTS

# 下载LJSpeech数据集（小样本，约100MB）
!wget https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
!tar -xjf LJSpeech-1.1.tar.bz2

# 预处理数据集（生成分词器文件）
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/ljspeech_pinyin \
    --tokenizer pinyin

/content/F5-TTS
--2026-03-12 08:24:34--  https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
Resolving data.keithito.com (data.keithito.com)... 169.150.249.167, 2400:52e0:1a01::1112:1
Connecting to data.keithito.com (data.keithito.com)|169.150.249.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2748572632 (2.6G) [text/plain]
Saving to: ‘LJSpeech-1.1.tar.bz2’

LJSpeech-1.1.tar.bz 100%[===================>]   2.56G   222MB/s    in 15s     

2026-03-12 08:24:49 (179 MB/s) - ‘LJSpeech-1.1.tar.bz2’ saved [2748572632/2748572632]


Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 68, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 21, in main
    with open(meta_info, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or 

In [ ]:
# 查看当前目录下有什么
!ls -la

# 查看LJSpeech-1.1文件夹内容
!ls -la LJSpeech-1.1/

total 2684232
drwxr-xr-x 8 root root       4096 Mar 12 08:24 .
drwxr-xr-x 1 root root       4096 Mar 12 07:07 ..
drwxr-xr-x 3 root root       4096 Mar 12 07:38 ckpts
drwxr-xr-x 3 root root       4096 Mar 12 07:00 data
-rw-r--r-- 1 root root        865 Mar 12 07:00 Dockerfile
drwxr-xr-x 8 root root       4096 Mar 12 07:00 .git
drwxr-xr-x 4 root root       4096 Mar 12 07:00 .github
-rw-r--r-- 1 root root       3202 Mar 12 07:00 .gitignore
-rw-r--r-- 1 root root        115 Mar 12 07:00 .gitmodules
-rw-r--r-- 1 root root       1068 Mar 12 07:00 LICENSE
drwxr-xr-x 3 1000 1000       4096 Feb 19  2018 LJSpeech-1.1
-rw-r--r-- 1 root root 2748572632 Feb 19  2018 LJSpeech-1.1.tar.bz2
-rw-r--r-- 1 root root        413 Mar 12 07:00 .pre-commit-config.yaml
-rw-r--r-- 1 root root       1561 Mar 12 07:00 pyproject.toml
-rw-r--r-- 1 root root       9759 Mar 12 07:00 README.md
-rw-r--r-- 1 root root        198 Mar 12 07:00 ruff.toml
drwxr-xr-x 5 root root       4096 Mar 12 07:00 src
total 3228
drwxr-xr

In [ ]:
# 先检查metadata.csv是否存在
import os
if os.path.exists("LJSpeech-1.1/metadata.csv"):
    print("✅ metadata.csv 存在")

    # 创建输出目录
    os.makedirs("data/ljspeech_pinyin", exist_ok=True)

    # 运行数据准备
    !python src/f5_tts/train/datasets/prepare_ljspeech.py \
        --data_dir ./LJSpeech-1.1 \
        --save_dir ./data/ljspeech_pinyin \
        --tokenizer pinyin \
        --use_char_tokenizer \
        --seed 42
else:
    print("❌ metadata.csv 不存在，请检查")
    !ls -la LJSpeech-1.1/

✅ metadata.csv 存在

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:07<00:00, 1800.86it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 292859.30it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours


In [ ]:
# 查看生成的数据文件
import os
data_dir = "/content/F5-TTS/data/LJSpeech_char"
print(f"数据目录内容：")
!ls -la {data_dir}

数据目录内容：
total 2156
drwxr-xr-x 2 root root    4096 Mar 12 08:37 .
drwxr-xr-x 5 root root    4096 Mar 12 08:37 ..
-rw-r--r-- 1 root root  248953 Mar 12 08:37 duration.json
-rw-r--r-- 1 root root 1942064 Mar 12 08:37 raw.arrow
-rw-r--r-- 1 root root     162 Mar 12 08:37 vocab.txt


In [ ]:
# 进入项目目录
%cd /content/F5-TTS

# 蒸馏测试命令（注意：!开头，并且不要有缩进）
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
copy checkpoint for finetune
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/../../data/LJSpeech_char_char/vocab.txt'


In [ ]:
# 1. 重命名数据集文件夹
%cd /content/F5-TTS
!mv data/LJSpeech_char data/LJSpeech_char_char

# 2. 验证重命名成功
!ls -la data/

# 3. 重新运行训练命令
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
total 240
drwxr-xr-x 5 root root   4096 Mar 12 08:41 .
drwxr-xr-x 8 root root   4096 Mar 12 08:24 ..
drwxr-xr-x 2 root root   4096 Mar 12 07:00 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 12 07:00 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 12 08:37 LJSpeech_char_char
drwxr-xr-x 2 root root   4096 Mar 12 08:31 ljspeech_pinyin
copy checkpoint for finetune
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such fi

In [ ]:
# 一键解决
%cd /content/F5-TTS

# 创建软链接
!mkdir -p /content/F5-TTS/src/f5_tts/../../data
!ln -sf /content/F5-TTS/data/LJSpeech_char_char /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char_char_char

# 验证
!ls -la /content/F5-TTS/src/f5_tts/../../data/

# 运行训练（注意dataset_name要用你实际的）
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
total 240
drwxr-xr-x 5 root root   4096 Mar 12 08:42 .
drwxr-xr-x 8 root root   4096 Mar 12 08:24 ..
drwxr-xr-x 2 root root   4096 Mar 12 07:00 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 12 07:00 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 12 08:37 LJSpeech_char_char
lrwxrwxrwx 1 root root     39 Mar 12 08:42 LJSpeech_char_char_char -> /content/F5-TTS/data/LJSpeech_char_char
drwxr-xr-x 2 root root   4096 Mar 12 08:31 ljspeech_pinyin

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape se

In [ ]:
%cd /content/F5-TTS

# 用 ignore_keys 忽略文本嵌入层
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples \
    --ignore_keys "text_embed"

/content/F5-TTS
usage: finetune_cli.py [-h] [--exp_name {F5TTS_v1_Base,F5TTS_Base,E2TTS_Base}]
                       [--dataset_name DATASET_NAME]
                       [--learning_rate LEARNING_RATE]
                       [--batch_size_per_gpu BATCH_SIZE_PER_GPU]
                       [--batch_size_type {frame,sample}]
                       [--max_samples MAX_SAMPLES]
                       [--grad_accumulation_steps GRAD_ACCUMULATION_STEPS]
                       [--max_grad_norm MAX_GRAD_NORM] [--epochs EPOCHS]
                       [--num_warmup_updates NUM_WARMUP_UPDATES]
                       [--save_per_updates SAVE_PER_UPDATES]
                       [--keep_last_n_checkpoints KEEP_LAST_N_CHECKPOINTS]
                       [--last_per_updates LAST_PER_UPDATES] [--finetune]
                       [--pretrain PRETRAIN]
                       [--tokenizer {pinyin,char,custom}]
                       [--tokenizer_path TOKENIZER_PATH] [--log_samples]
                       [

In [ ]:
%cd /content/F5-TTS

# 不加载教师模型，只测试学生模型能否训练
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1860537.13it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2692741.11it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-

In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1649219.07it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2422208.71it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-

In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1854759.06it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2752223.12it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-

In [ ]:
%cd /content/F5-TTS

# 查看现有的检查点
!ls -la ckpts/F5TTS_Base/

# 删除所有检查点（或者重命名备份）
!rm -rf ckpts/F5TTS_Base/*

# 确认删除成功
!ls -la ckpts/F5TTS_Base/

# 现在重新运行训练
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --tokenizer char \
    --log_samples

/content/F5-TTS
ls: cannot access 'ckpts/F5TTS_Base/': No such file or directory
ls: cannot access 'ckpts/F5TTS_Base/': No such file or directory

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1061767.04it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 1437495.29it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most r

In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/train.py \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size 4 \
    --epochs 1 \
    --tokenizer char

/content/F5-TTS
usage: train.py [--help] [--hydra-help] [--version] [--cfg {job,hydra,all}]
                [--resolve] [--package PACKAGE] [--run] [--multirun]
                [--shell-completion] [--config-path CONFIG_PATH]
                [--config-name CONFIG_NAME] [--config-dir CONFIG_DIR]
                [--experimental-rerun EXPERIMENTAL_RERUN]
                [--info [{all,config,defaults,defaults-tree,plugins,searchpath}]]
                [overrides ...]
train.py: error: unrecognized arguments: --dataset_name --learning_rate 7.5e-5 --batch_size 4 --epochs 1 --tokenizer char


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1854571.25it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2738642.40it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-

In [ ]:
%cd /content/F5-TTS

# 确保检查点目录为空
!mkdir -p ckpts/F5TTS_Base/
!rm -f ckpts/F5TTS_Base/*

# 运行训练
!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --tokenizer char \
    --log_samples

[Errno 2] No such file or directory: '/content/F5-TTS'
/content
python3: can't open file '/content/src/f5_tts/train/finetune_cli.py': [Errno 2] No such file or directory


In [ ]:
# 克隆项目
!git clone https://github.com/SWivid/F5-TTS.git
%cd /content/F5-TTS

# 安装依赖
!pip install -e .

Cloning into 'F5-TTS'...
remote: Enumerating objects: 3955, done.
remote: Counting objects: 100% (262/262), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 3955 (delta 185), reused 113 (delta 113), pack-reused 3693 (from 3)
Receiving objects: 100% (3955/3955), 2.38 MiB | 26.53 MiB/s, done.
Resolving deltas: 100% (2383/2383), done.
/content/F5-TTS
Obtaining file:///content/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 81.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 16.1 MB/s 

In [ ]:
# 确认教师模型存在
!ls -la /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/

ls: cannot access '/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/': No such file or directory


In [ ]:
# 1. 先确认你的 Google Drive 挂载是否正常
from google.colab import drive
drive.mount('/content/drive')

# 2. 查看你的 f5tts_project 目录下有什么
!ls -la /content/drive/MyDrive/f5tts_project/

# 3. 如果存在，继续深入查看
!ls -la /content/drive/MyDrive/f5tts_project/models/

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Mar 12 07:07 cache
drwx------ 2 root root 4096 Mar 12 07:07 models
drwx------ 2 root root 4096 Mar 12 07:07 output
total 4
drwx------ 2 root root 4096 Mar 12 07:21 F5TTS-Base


In [ ]:
# 1. 在 Google Drive 中创建数据目录
import os
drive_data_path = "/content/drive/MyDrive/f5tts_project/datasets"
os.makedirs(drive_data_path, exist_ok=True)

# 2. 切换到 Drive 目录下载
%cd {drive_data_path}

# 3. 下载数据集到 Drive（永久保存）
!wget https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2

# 4. 解压到 Drive（解压后约 2.6GB）
!tar -xjf LJSpeech-1.1.tar.bz2

# 5. 验证下载成功
!ls -la {drive_data_path}/LJSpeech-1.1/

/content/drive/MyDrive/f5tts_project/datasets
--2026-03-16 13:24:02--  https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
Resolving data.keithito.com (data.keithito.com)... 143.244.49.179, 2400:52e0:1a01::1001:1
Connecting to data.keithito.com (data.keithito.com)|143.244.49.179|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2748572632 (2.6G) [text/plain]
Saving to: ‘LJSpeech-1.1.tar.bz2’

LJSpeech-1.1.tar.bz 100%[===================>]   2.56G  64.4MB/s    in 45s     

2026-03-16 13:24:47 (58.4 MB/s) - ‘LJSpeech-1.1.tar.bz2’ saved [2748572632/2748572632]

^C
total 2707
-rw------- 1 root root 2767490 Jul  5  2017 metadata.csv
drwx------ 2 root root    4096 Mar 16 13:31 wavs


In [ ]:
# 检查 wavs 文件夹内容
%cd /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1

# 1. 查看 wavs 文件夹大小（应该约 2.6GB）
!du -sh wavs/

# 2. 统计 wav 文件数量（应该是 13100 个）
!ls -1 wavs/ | wc -l

# 3. 查看前几个文件确认
!ls -la wavs/ | head -10

/content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1
3.2G	wavs/
11684
total 3308115
-rw------- 1 root root 425830 Jul  4  2017 LJ001-0001.wav
-rw------- 1 root root  83814 Jul  4  2017 LJ001-0002.wav
-rw------- 1 root root 426342 Jul  4  2017 LJ001-0003.wav
-rw------- 1 root root 226662 Jul  4  2017 LJ001-0004.wav
-rw------- 1 root root 357734 Jul  4  2017 LJ001-0005.wav
-rw------- 1 root root 250726 Jul  4  2017 LJ001-0006.wav
-rw------- 1 root root 370022 Jul  4  2017 LJ001-0007.wav
-rw------- 1 root root  78694 Jul  4  2017 LJ001-0008.wav
-rw------- 1 root root 333158 Jul  4  2017 LJ001-0009.wav


In [ ]:
# 重新解压
%cd /content/drive/MyDrive/f5tts_project/datasets

# 删除不完整的解压目录
!rm -rf LJSpeech-1.1

/content/drive/MyDrive/f5tts_project/datasets


In [ ]:
# 一键解决
%cd /content

# 复制压缩包到本地（更快）
!cp /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1.tar.bz2 ./

/content


In [ ]:
# 1. 先按方案一下载到 Drive

# 2. 在项目目录创建软链接
%cd /content/F5-TTS
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 ./LJSpeech-1.1

# 3. 验证链接
!ls -la ./LJSpeech-1.1/

[Errno 2] No such file or directory: '/content/F5-TTS'
/content
total 2828
drwx------ 3 root root    4096 Mar 16 13:38 .
drwxr-xr-x 1 root root    4096 Mar 16 13:35 ..
lrwxrwxrwx 1 root root      58 Mar 16 13:38 LJSpeech-1.1 -> /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1
-rw-r--r-- 1 1000 1000 2767490 Jul  5  2017 metadata.csv
drwx------ 2 root root  114688 Mar 16 13:36 wavs


In [ ]:
# 1. 重新克隆 F5-TTS
!git clone https://github.com/SWivid/F5-TTS.git
%cd /content/F5-TTS

Cloning into 'F5-TTS'...
remote: Enumerating objects: 3955, done.
remote: Counting objects: 100% (262/262), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 3955 (delta 185), reused 113 (delta 113), pack-reused 3693 (from 3)
Receiving objects: 100% (3955/3955), 2.38 MiB | 25.96 MiB/s, done.
Resolving deltas: 100% (2383/2383), done.
/content/F5-TTS


In [ ]:
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 /content/F5-TTS/LJSpeech-1.1


In [ ]:
!pip install -e .

Obtaining file:///content/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 86.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 3. 现在重新运行数据准备
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_char_char \
    --tokenizer char


Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 68, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 21, in main
    with open(meta_info, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '<SOME_PATH>/LJSpeech-1.1/metadata.csv'


In [ ]:
ls /content

drive/  F5-TTS/  LJSpeech-1.1/  LJSpeech-1.1.tar.bz2  sample_data/


In [ ]:
ls /content/LJSpeech-1.1

LJSpeech-1.1@  metadata.csv  wavs/


In [ ]:
nano src/f5_tts/train/datasets/prepare_ljspeech.py

SyntaxError: invalid syntax (1644657058.py, line 1)

In [ ]:
# 一键修复并运行
%cd /content/F5-TTS

# 修复脚本
file_path = "src/f5_tts/train/datasets/prepare_ljspeech.py"
with open(file_path, "r") as f:
    content = f.read()
content = content.replace("<SOME_PATH>/LJSpeech-1.1", "./LJSpeech-1.1")
with open(file_path, "w") as f:
    f.write(content)
print("✅ 脚本修复完成")

# 运行数据准备
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_char_char \
    --tokenizer char

/content/F5-TTS
✅ 脚本修复完成

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 68, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 21, in main
    with open(meta_info, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: './LJSpeech-1.1/metadata.csv'


In [ ]:
!pwd
!ls -la


/content
total 20
drwxr-xr-x 1 root root 4096 Mar 17 09:25 .
drwxr-xr-x 1 root root 4096 Mar 17 09:07 ..
drwxr-xr-x 4 root root 4096 Feb  6 14:31 .config
drwx------ 5 root root 4096 Mar 17 09:25 drive
drwxr-xr-x 1 root root 4096 Feb  6 14:31 sample_data


In [ ]:
# 直接进入 datasets 目录
%cd /content/F5-TTS/src/f5_tts/train/datasets/



[Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/train/datasets/'
/content


In [ ]:
# ==================== 完整恢复 ====================
%cd /content

# 1. 重新克隆 F5-TTS
!git clone https://github.com/SWivid/F5-TTS.git

# 2. 进入项目目录
%cd /content/F5-TTS

# 3. 安装依赖（重要！）
!pip install -e .

# 4. 恢复数据软链接（链接到你 Drive 里的数据集）
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 ./LJSpeech-1.1

# 5. 确认一切就绪
!ls -la
!ls -la LJSpeech-1.1/ | head -5

print("✅ 恢复完成！")

/content
Cloning into 'F5-TTS'...
remote: Enumerating objects: 3955, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 3955 (delta 185), reused 111 (delta 111), pack-reused 3694 (from 3)
Receiving objects: 100% (3955/3955), 2.38 MiB | 6.75 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/content/F5-TTS
Obtaining file:///content/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 76.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 19

In [ ]:
# 确认你在正确目录
%cd /content/F5-TTS
!pwd
!ls -la

/content/F5-TTS
/content/F5-TTS
total 68
drwxr-xr-x 7 root root 4096 Mar 17 09:30 .
drwxr-xr-x 1 root root 4096 Mar 17 09:29 ..
drwxr-xr-x 2 root root 4096 Mar 17 09:29 ckpts
drwxr-xr-x 3 root root 4096 Mar 17 09:29 data
-rw-r--r-- 1 root root  865 Mar 17 09:29 Dockerfile
drwxr-xr-x 8 root root 4096 Mar 17 09:29 .git
drwxr-xr-x 4 root root 4096 Mar 17 09:29 .github
-rw-r--r-- 1 root root 3202 Mar 17 09:29 .gitignore
-rw-r--r-- 1 root root  115 Mar 17 09:29 .gitmodules
-rw-r--r-- 1 root root 1068 Mar 17 09:29 LICENSE
lrwxrwxrwx 1 root root   58 Mar 17 09:30 LJSpeech-1.1 -> /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1
-rw-r--r-- 1 root root  413 Mar 17 09:29 .pre-commit-config.yaml
-rw-r--r-- 1 root root 1561 Mar 17 09:29 pyproject.toml
-rw-r--r-- 1 root root 9759 Mar 17 09:29 README.md
-rw-r--r-- 1 root root  198 Mar 17 09:29 ruff.toml
drwxr-xr-x 5 root root 4096 Mar 17 09:29 src


In [ ]:
# 检查 LJSpeech 数据集是否链接成功
!ls -la LJSpeech-1.1/
!ls -la LJSpeech-1.1/wavs/ | head -5

ls: cannot access 'LJSpeech-1.1/': No such file or directory
ls: cannot access 'LJSpeech-1.1/wavs/': No such file or directory


In [ ]:
# 2. 进入 F5-TTS 目录
%cd /content/F5-TTS

# 3. 创建软链接（使用绝对路径）
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 ./LJSpeech-1.1

# 4. 验证链接是否成功
!ls -la ./LJSpeech-1.1/
!ls -la ./LJSpeech-1.1/wavs/ | head -5

/content/F5-TTS
ls: cannot access './LJSpeech-1.1/': No such file or directory
ls: cannot access './LJSpeech-1.1/wavs/': No such file or directory


In [ ]:
# 搜索数据集
%cd /content
!find /content/drive/MyDrive -name "metadata.csv" 2>/dev/null

/content


In [ ]:
# 查找 metadata.csv 文件
!find /content/drive/MyDrive -name "metadata.csv" 2>/dev/null

In [ ]:
# 进入数据集目录
%cd /content/drive/MyDrive/f5tts_project/datasets/

# 解压（显示进度）
!tar -xjvf LJSpeech-1.1.tar.bz2

# 验证解压成功
!ls -la LJSpeech-1.1/

流式输出内容被截断，只能显示最后 5000 行内容。
LJSpeech-1.1/wavs/LJ034-0035.wav
LJSpeech-1.1/wavs/LJ010-0152.wav
LJSpeech-1.1/wavs/LJ036-0174.wav
LJSpeech-1.1/wavs/LJ035-0076.wav
LJSpeech-1.1/wavs/LJ032-0176.wav
LJSpeech-1.1/wavs/LJ046-0113.wav
LJSpeech-1.1/wavs/LJ017-0096.wav
LJSpeech-1.1/wavs/LJ004-0098.wav
LJSpeech-1.1/wavs/LJ010-0147.wav
LJSpeech-1.1/wavs/LJ042-0230.wav
LJSpeech-1.1/wavs/LJ041-0033.wav
LJSpeech-1.1/wavs/LJ045-0229.wav
LJSpeech-1.1/wavs/LJ014-0199.wav
LJSpeech-1.1/wavs/LJ002-0082.wav
LJSpeech-1.1/wavs/LJ006-0055.wav
LJSpeech-1.1/wavs/LJ045-0120.wav
LJSpeech-1.1/wavs/LJ050-0028.wav
LJSpeech-1.1/wavs/LJ045-0215.wav
LJSpeech-1.1/wavs/LJ013-0121.wav
LJSpeech-1.1/wavs/LJ008-0025.wav
LJSpeech-1.1/wavs/LJ005-0240.wav
LJSpeech-1.1/wavs/LJ044-0026.wav
LJSpeech-1.1/wavs/LJ048-0127.wav
LJSpeech-1.1/wavs/LJ006-0195.wav
LJSpeech-1.1/wavs/LJ030-0151.wav
LJSpeech-1.1/wavs/LJ038-0154.wav
LJSpeech-1.1/wavs/LJ003-0174.wav
LJSpeech-1.1/wavs/LJ003-0250.wav
LJSpeech-1.1/wavs/LJ027-0147.wav
LJSpeech-1.1/wav

In [ ]:
# 1. 进入 F5-TTS 目录
%cd /content/F5-TTS

# 2. 创建软链接（指向 Drive 里的数据集）
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 ./LJSpeech-1.1

# 3. 验证链接成功
!ls -la ./LJSpeech-1.1/
!ls -la ./LJSpeech-1.1/wavs/ | head -5

/content/F5-TTS
total 2712
lrw------- 1 root root       0 Mar 17 09:49 LJSpeech-1.1 -> /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1
-rw------- 1 root root 2767490 Jul  5  2017 metadata.csv
-rw------- 1 root root    4486 Feb 19  2018 README
drwx------ 2 root root    4096 Feb 19  2018 wavs
total 3711286
-rw------- 1 root root 425830 Jul  4  2017 LJ001-0001.wav
-rw------- 1 root root  83814 Jul  4  2017 LJ001-0002.wav
-rw------- 1 root root 426342 Jul  4  2017 LJ001-0003.wav
-rw------- 1 root root 226662 Jul  4  2017 LJ001-0004.wav


In [ ]:
# 4. 运行数据准备
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_char_char \
    --tokenizer char


Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 68, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/datasets/prepare_ljspeech.py", line 21, in main
    with open(meta_info, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '<SOME_PATH>/LJSpeech-1.1/metadata.csv'


In [ ]:
# 确认软链接能正确指向 Drive
%cd /content/F5-TTS
!ls -la ./LJSpeech-1.1/metadata.csv

/content/F5-TTS
-rw------- 1 root root 2767490 Jul  5  2017 ./LJSpeech-1.1/metadata.csv


In [ ]:
#准备数据
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_char_char \
    --tokenizer char


Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:49<00:00, 264.22it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 232398.93it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours


In [ ]:
# 1. 确认教师模型路径
teacher_path = "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt"

# 2. 检查文件是否存在
import os
if os.path.exists(teacher_path):
    print(f"✅ 教师模型存在：{teacher_path}")
else:
    print("❌ 教师模型不存在，请检查路径")

✅ 教师模型存在：/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
copy checkpoint for finetune
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/../../data/LJSpeech_char_char/vocab.txt'


In [ ]:
!ls -la data/

total 236
drwxr-xr-x 4 root root   4096 Mar 17 09:56 .
drwxr-xr-x 7 root root   4096 Mar 17 09:33 ..
drwxr-xr-x 2 root root   4096 Mar 17 09:29 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 17 09:29 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 17 09:56 LJSpeech_char


In [ ]:
# 更简单：直接重命名
%cd /content/F5-TTS
!mv data/LJSpeech_char data/LJSpeech_char_char

# 验证
!ls -la data/

/content/F5-TTS
total 236
drwxr-xr-x 4 root root   4096 Mar 17 10:06 .
drwxr-xr-x 7 root root   4096 Mar 17 09:33 ..
drwxr-xr-x 2 root root   4096 Mar 17 09:29 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 17 09:29 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 17 09:56 LJSpeech_char_char


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
Download Vocos from huggingface charactr/vocos-mel-24khz
config.yaml: 100% 461/461 [00:00<00:00, 2.24MB/s]
pytorch_model.bin: 100% 54.4M/54.4M [00:01<00:00, 52.8MB/s]
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100

In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples \
    --ignore_keys "text_embed"

/content/F5-TTS
usage: finetune_cli.py [-h] [--exp_name {F5TTS_v1_Base,F5TTS_Base,E2TTS_Base}]
                       [--dataset_name DATASET_NAME]
                       [--learning_rate LEARNING_RATE]
                       [--batch_size_per_gpu BATCH_SIZE_PER_GPU]
                       [--batch_size_type {frame,sample}]
                       [--max_samples MAX_SAMPLES]
                       [--grad_accumulation_steps GRAD_ACCUMULATION_STEPS]
                       [--max_grad_norm MAX_GRAD_NORM] [--epochs EPOCHS]
                       [--num_warmup_updates NUM_WARMUP_UPDATES]
                       [--save_per_updates SAVE_PER_UPDATES]
                       [--keep_last_n_checkpoints KEEP_LAST_N_CHECKPOINTS]
                       [--last_per_updates LAST_PER_UPDATES] [--finetune]
                       [--pretrain PRETRAIN]
                       [--tokenizer {pinyin,char,custom}]
                       [--tokenizer_path TOKENIZER_PATH] [--log_samples]
                       [

In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_char \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  75

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1152112.19it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 1453773.84it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-

In [ ]:
# 检查官方词汇表
%cd /content/F5-TTS
!ls -la src/f5_tts/infer/examples/vocab.txt
!wc -l src/f5_tts/infer/examples/vocab.txt  # 应该显示2545行

/content/F5-TTS
-rw-r--r-- 1 root root 11255 Mar 17 09:29 src/f5_tts/infer/examples/vocab.txt
2545 src/f5_tts/infer/examples/vocab.txt


In [ ]:
# 使用官方词汇表重新生成数据
%cd /content/F5-TTS

!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_official_vocab \
    --tokenizer char \
    --vocab_file src/f5_tts/infer/examples/vocab.txt  # 指定官方词表

/content/F5-TTS

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:45<00:00, 289.38it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 393135.35it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours


In [ ]:
# ==================== Colab断联后一键恢复 ====================

# 1. 重新挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. 重新克隆F5-TTS（代码会丢失，但数据在Drive里）
!git clone https://github.com/SWivid/F5-TTS.git
%cd /content/F5-TTS

# 3. 安装依赖
!pip install -e .

# 4. 恢复数据软链接（链接到Drive里的数据集）
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 ./LJSpeech-1.1

# 5. 确认数据还在
!ls -la ./LJSpeech-1.1/
!ls -la ./LJSpeech-1.1/wavs/ | head -5

print("✅ 环境恢复完成！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'F5-TTS'...
remote: Enumerating objects: 3955, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 3955 (delta 185), reused 111 (delta 111), pack-reused 3694 (from 3)
Receiving objects: 100% (3955/3955), 2.38 MiB | 7.03 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/content/F5-TTS
Obtaining file:///content/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for f5-tts (pyproject.toml) ... done
  Created wheel for f5-tts: filename=f5_tts-1.1.17-0.editable-py3-none-any.whl size=6831 sha256=25cebb0a2f09391c7f3602d52fce3accfc1c3a38dfb1ad451c4e43c8b7b45b14
  Stored in directory: /tmp/pip-ephem-whe

In [ ]:
!ls /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt

/content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt


In [ ]:
# 运行此命令来查看当前目录结构
!ls /content/F5-TTS

ckpts  Dockerfile  LICENSE	 pyproject.toml  ruff.toml
data   F5-TTS	   LJSpeech-1.1  README.md	 src


In [ ]:
%cd /content/F5-TTS

# 删除之前生成的数据
!rm -rf ./data/LJSpeech_char
!rm -rf ./data/LJSpeech_char_char
!rm -rf ./data/LJSpeech_official_vocab

# 确认删除
!ls -la ./data/

/content/F5-TTS
total 232
drwxr-xr-x 3 root root   4096 Mar 17 11:59 .
drwxr-xr-x 8 root root   4096 Mar 17 11:50 ..
drwxr-xr-x 2 root root   4096 Mar 17 09:29 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 17 09:29 librispeech_pc_test_clean_cross_sentence.lst


In [ ]:
%cd /content/F5-TTS

# 用官方词vocab表生成数据
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_2546vocab \
    --tokenizer char \
    --vocab_file /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt

/content/F5-TTS

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:38<00:00, 340.43it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 301945.82it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours


In [ ]:
!ls /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt

/content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt


In [ ]:
%cd /content/F5-TTS

# 1. 先正常运行数据准备（生成75词表的数据）
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_2546vocab_temp \
    --tokenizer char

# 2. 用官方词表替换生成的 vocab.txt
!cp /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt ./data/LJSpeech_2546vocab_temp/vocab.txt

# 3. 重命名为最终目录
!mv ./data/LJSpeech_2546vocab_temp ./data/LJSpeech_2546vocab

# 4. 验证
!wc -l ./data/LJSpeech_2546vocab/vocab.txt

/content/F5-TTS

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:37<00:00, 344.77it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 495409.59it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours
cp: cannot create regular file './data/LJSpeech_2546vocab_temp/vocab.txt': No such file or directory
mv: cannot stat './data/LJSpeech_2546vocab_temp': No such file or directory
wc: ./data/LJSpeech_2546vocab/vocab.txt: No such file or directory


In [ ]:
%cd /content/F5-TTS

# 1. 先找到实际生成的数据在哪里
!ls -la /content/F5-TTS/src/f5_tts/../../data/

# 2. 创建目标目录
!mkdir -p ./data/LJSpeech_2546vocab

# 3. 把生成的文件复制过去
!cp /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char/raw.arrow ./data/LJSpeech_2546vocab/
!cp /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char/duration.json ./data/LJSpeech_2546vocab/

# 4. 用官方词表替换 vocab.txt
!cp /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt ./data/LJSpeech_2546vocab/vocab.txt

# 5. 验证
!ls -la ./data/LJSpeech_2546vocab/
!wc -l ./data/LJSpeech_2546vocab/vocab.txt

/content/F5-TTS
total 236
drwxr-xr-x 4 root root   4096 Mar 17 12:00 .
drwxr-xr-x 8 root root   4096 Mar 17 11:50 ..
drwxr-xr-x 2 root root   4096 Mar 17 09:29 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 17 09:29 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 17 12:00 LJSpeech_char
total 2168
drwxr-xr-x 2 root root    4096 Mar 17 12:09 .
drwxr-xr-x 5 root root    4096 Mar 17 12:09 ..
-rw-r--r-- 1 root root  248953 Mar 17 12:09 duration.json
-rw-r--r-- 1 root root 1942064 Mar 17 12:09 raw.arrow
-rw-r--r-- 1 root root   13800 Mar 17 12:09 vocab.txt
2545 ./data/LJSpeech_2546vocab/vocab.txt


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
copy checkpoint for finetune
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/../../data/LJSpeech_2546vocab_char/vocab.txt'


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_2546vocab \
    --tokenizer char

/content/F5-TTS

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:38<00:00, 342.50it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 517016.23it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours


In [ ]:
%cd /content/F5-TTS

# 创建目标目录
!mkdir -p ./data/LJSpeech_2546vocab

/content/F5-TTS


In [ ]:
# 复制生成的raw.arrow和duration.json
!cp /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char/raw.arrow ./data/LJSpeech_2546vocab/
!cp /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char/duration.json ./data/LJSpeech_2546vocab/


In [ ]:
# 验证复制成功
!ls -la ./data/LJSpeech_2546vocab/

total 2168
drwxr-xr-x 2 root root    4096 Mar 17 12:09 .
drwxr-xr-x 5 root root    4096 Mar 17 12:09 ..
-rw-r--r-- 1 root root  248953 Mar 17 12:15 duration.json
-rw-r--r-- 1 root root 1942064 Mar 17 12:15 raw.arrow
-rw-r--r-- 1 root root   13800 Mar 17 12:09 vocab.txt


In [ ]:
%cd /content/F5-TTS

# 1. 先查看官方词表大小
!wc -l /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt

/content/F5-TTS
2545 /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt


In [ ]:
# 2. 用官方词表替换
!cp /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt ./data/LJSpeech_2546vocab/vocab.txt

In [ ]:
# 3. 验证替换后的词表
!wc -l ./data/LJSpeech_2546vocab/vocab.txt
!head -5 ./data/LJSpeech_2546vocab/vocab.txt

2545 ./data/LJSpeech_2546vocab/vocab.txt
 
!
"
#
$


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/../../data/LJSpeech_2546vocab_char/vocab.txt'


In [ ]:
%cd /content/F5-TTS

# 查看当前数据目录
!ls -la ./data/

# 把 LJSpeech_2546vocab 改名为 LJSpeech_2546vocab_char
!mv ./data/LJSpeech_2546vocab ./data/LJSpeech_2546vocab_char

/content/F5-TTS
total 240
drwxr-xr-x 5 root root   4096 Mar 17 12:09 .
drwxr-xr-x 8 root root   4096 Mar 17 11:50 ..
drwxr-xr-x 2 root root   4096 Mar 17 09:29 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 17 09:29 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 17 12:09 LJSpeech_2546vocab
drwxr-xr-x 2 root root   4096 Mar 17 12:00 LJSpeech_char


In [ ]:
# 验证改名成功
!ls -la ./data/
!ls -la ./data/LJSpeech_2546vocab_char/

total 240
drwxr-xr-x 5 root root   4096 Mar 17 12:18 .
drwxr-xr-x 8 root root   4096 Mar 17 11:50 ..
drwxr-xr-x 2 root root   4096 Mar 17 09:29 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 17 09:29 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 17 12:09 LJSpeech_2546vocab_char
drwxr-xr-x 2 root root   4096 Mar 17 12:00 LJSpeech_char
total 2168
drwxr-xr-x 2 root root    4096 Mar 17 12:09 .
drwxr-xr-x 5 root root    4096 Mar 17 12:18 ..
-rw-r--r-- 1 root root  248953 Mar 17 12:15 duration.json
-rw-r--r-- 1 root root 1942064 Mar 17 12:15 raw.arrow
-rw-r--r-- 1 root root   13800 Mar 17 12:16 vocab.txt


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 32 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

    #自己的备注：下方print vocab为2545 对了！！！！

/content/F5-TTS

vocab :  2545

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1564058.71it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2408406.35it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F

In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 5000 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  2545

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1941876.03it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2574760.19it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F

In [ ]:
# 使用全部数据# 只跑1个epoch# 每500步保存一次

%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples -1 \
    --epochs 1 \
    --save_per_updates 500 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples


/content/F5-TTS

vocab :  2545

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1677158.28it/s]
Creating dynamic batches with 4 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2357867.33it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F

In [ ]:
# 直接修改 trainer.py，注释掉整个恢复机制
file_path = "/content/F5-TTS/src/f5_tts/model/trainer.py"

with open(file_path, "r") as f:
    lines = f.readlines()

# 找到 train 函数中关于恢复的部分，直接注释掉
for i, line in enumerate(lines):
    if "if exists(resumable_with_seed):" in line:
        print(f"找到恢复代码在第 {i+1} 行")
        # 注释掉整个 if 块
        lines[i] = "        # 禁用恢复机制\n"
        lines[i+1] = "        skipped_epoch = 0\n"
        lines[i+2] = "        skipped_batch = 0\n"
        # 注释掉后面的代码
        for j in range(i+3, i+10):
            if j < len(lines):
                lines[j] = "# " + lines[j]
        break

# 同时确保 orig_epoch_step 有默认值
for i, line in enumerate(lines):
    if "orig_epoch_step = len(train_dataloader)" in line:
        lines[i] = "        orig_epoch_step = len(train_dataloader) or 1  # 防止除以0\n"
        print(f"已修改 orig_epoch_step 行")

with open(file_path, "w") as f:
    f.writelines(lines)

print("✅ 修改完成，现在可以运行训练")

找到恢复代码在第 276 行
已修改 orig_epoch_step 行
✅ 修改完成，现在可以运行训练


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 4 \
    --max_samples 128 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 8, in <module>
    from f5_tts.model import CFM, DiT, Trainer, UNetT
  File "/content/F5-TTS/src/f5_tts/model/__init__.py", line 5, in <module>
    from f5_tts.model.trainer import Trainer
  File "/content/F5-TTS/src/f5_tts/model/trainer.py", line 286
    num_workers=num_workers,
IndentationError: unexpected indent


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 3800 \
    --max_samples 128 \
    --epochs 1 \
    --save_per_updates 10 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples \


/content/F5-TTS

vocab :  2545

vocoder :  vocos
Using logger: None
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/13100 [00:00<00:00, 1791444.11it/s]
Creating dynamic batches with 3800 audio frames per gpu: 100% 13100/13100 [00:00<00:00, 2409145.54it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1/1:   0% 0/2318 [00:00<?, ?update/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataL

In [ ]:
# ==================== Colab断联后一键恢复 ====================

# 1. 重新挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. 重新克隆F5-TTS（代码会丢失，数据在Drive里）
!git clone https://github.com/SWivid/F5-TTS.git
%cd /content/F5-TTS

# 3. 安装依赖
!pip install -e .

# 4. 恢复数据软链接（链接到Drive里的数据集和processed数据）
!ln -sf /content/drive/MyDrive/f5tts_project/datasets/LJSpeech-1.1 ./LJSpeech-1.1


# 5. 确认教师模型存在
!ls -la /content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt

print("✅ 环境恢复完成！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'F5-TTS'...
remote: Enumerating objects: 3955, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 3955 (delta 185), reused 111 (delta 111), pack-reused 3694 (from 3)
Receiving objects: 100% (3955/3955), 2.38 MiB | 6.43 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/content/F5-TTS
Obtaining file:///content/F5-TTS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━

In [ ]:
# 检查是否备份到 Drive
!ls -la /content/drive/MyDrive/f5tts_project/configs/

ls: cannot access '/content/drive/MyDrive/f5tts_project/configs/': No such file or directory


In [ ]:
# 创建蒸馏配置文件并备份到 Drive
import os

# 读取原有的Small配置作为基础
small_config_path = "/content/F5-TTS/src/f5_tts/configs/F5TTS_Small.yaml"
with open(small_config_path, 'r') as f:
    small_config = f.read()

# 蒸馏参数（注意缩进）
distill_config = small_config + """

# ===== 蒸馏参数 =====
distillation:
  teacher_model_path: "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base"
  temperature: 4.0
  alpha: 0.7

# 输出到Drive
hydra:
  run:
    dir: /content/drive/MyDrive/f5tts_project/output/distill_${now:%Y-%m-%d_%H-%M-%S}
"""

# 1. 保存到 Colab 本地
new_config_path = "/content/F5-TTS/src/f5_tts/configs/F5TTS_Small_Distill.yaml"
with open(new_config_path, 'w') as f:
    f.write(distill_config)
print(f"✅ 已保存到 Colab: {new_config_path}")

# 2. 备份到 Google Drive
drive_config_dir = "/content/drive/MyDrive/f5tts_project/configs"
os.makedirs(drive_config_dir, exist_ok=True)
drive_config_path = f"{drive_config_dir}/F5TTS_Small_Distill.yaml"
with open(drive_config_path, 'w') as f:
    f.write(distill_config)
print(f"✅ 已备份到 Drive: {drive_config_path}")

✅ 已保存到 Colab: /content/F5-TTS/src/f5_tts/configs/F5TTS_Small_Distill.yaml
✅ 已备份到 Drive: /content/drive/MyDrive/f5tts_project/configs/F5TTS_Small_Distill.yaml


In [ ]:
# 检查数据是否存在
%cd /content/F5-TTS

# 1. 检查数据目录
!ls -la ./data/LJSpeech_2546vocab/

# 2. 检查三个关键文件
!ls -la ./data/LJSpeech_2546vocab/raw.arrow
!ls -la ./data/LJSpeech_2546vocab/duration.json
!ls -la ./data/LJSpeech_2546vocab/vocab.txt

# 3. 检查词表大小（应该是2545）
!wc -l ./data/LJSpeech_2546vocab/vocab.txt

/content/F5-TTS
ls: cannot access './data/LJSpeech_2546vocab/': No such file or directory
ls: cannot access './data/LJSpeech_2546vocab/raw.arrow': No such file or directory
ls: cannot access './data/LJSpeech_2546vocab/duration.json': No such file or directory
ls: cannot access './data/LJSpeech_2546vocab/vocab.txt': No such file or directory
wc: ./data/LJSpeech_2546vocab/vocab.txt: No such file or directory


In [ ]:
%cd /content/F5-TTS

# 重新生成数据
!python src/f5_tts/train/datasets/prepare_ljspeech.py \
    --data_dir ./LJSpeech-1.1 \
    --save_dir ./data/LJSpeech_2546vocab \
    --tokenizer char \
    --vocab_file /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt



/content/F5-TTS

Prepare for LJSpeech_char, will save to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char

100% 13100/13100 [00:38<00:00, 336.61it/s]

Saving to /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char ...
Writing to raw.arrow ...: 100% 13100/13100 [00:00<00:00, 381517.47it/s]

For LJSpeech_char, sample count: 13100
For LJSpeech_char, vocab size is: 75
For LJSpeech_char, total 23.92 hours


In [ ]:
%cd /content/F5-TTS

# 1. 查看生成的数据在哪里
!ls -la /content/F5-TTS/src/f5_tts/../../data/

# 2. 创建目标目录
!mkdir -p ./data/LJSpeech_2546vocab

# 3. 把生成的文件复制过去
!cp /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char/raw.arrow ./data/LJSpeech_2546vocab/
!cp /content/F5-TTS/src/f5_tts/../../data/LJSpeech_char/duration.json ./data/LJSpeech_2546vocab/

# 4. 用官方词表替换 vocab.txt
!cp /content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt ./data/LJSpeech_2546vocab/vocab.txt

# 5. 验证
!ls -la ./data/LJSpeech_2546vocab/
!wc -l ./data/LJSpeech_2546vocab/vocab.txt

/content/F5-TTS
total 236
drwxr-xr-x 4 root root   4096 Mar 23 07:51 .
drwxr-xr-x 7 root root   4096 Mar 23 07:14 ..
drwxr-xr-x 2 root root   4096 Mar 23 07:13 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 23 07:13 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 23 07:51 LJSpeech_char
total 2168
drwxr-xr-x 2 root root    4096 Mar 23 07:53 .
drwxr-xr-x 5 root root    4096 Mar 23 07:53 ..
-rw-r--r-- 1 root root  248953 Mar 23 07:53 duration.json
-rw-r--r-- 1 root root 1942064 Mar 23 07:53 raw.arrow
-rw-r--r-- 1 root root   13800 Mar 23 07:53 vocab.txt
2545 ./data/LJSpeech_2546vocab/vocab.txt


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 3800 \
    --max_samples 128 \
    --epochs 1 \
    --save_per_updates 1000 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS
copy checkpoint for finetune
Traceback (most recent call last):
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 214, in <module>
    main()
  File "/content/F5-TTS/src/f5_tts/train/finetune_cli.py", line 163, in main
    vocab_char_map, vocab_size = get_tokenizer(tokenizer_path, tokenizer)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/F5-TTS/src/f5_tts/model/utils.py", line 124, in get_tokenizer
    with open(tokenizer_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/F5-TTS/src/f5_tts/../../data/LJSpeech_2546vocab_char/vocab.txt'


In [ ]:
%cd /content/F5-TTS

# 重命名数据目录
!mv ./data/LJSpeech_2546vocab ./data/LJSpeech_2546vocab_char

# 验证
!ls -la ./data/
!ls -la ./data/LJSpeech_2546vocab_char/

/content/F5-TTS
total 240
drwxr-xr-x 5 root root   4096 Mar 23 07:57 .
drwxr-xr-x 7 root root   4096 Mar 23 07:14 ..
drwxr-xr-x 2 root root   4096 Mar 23 07:13 Emilia_ZH_EN_pinyin
-rw-r--r-- 1 root root 222318 Mar 23 07:13 librispeech_pc_test_clean_cross_sentence.lst
drwxr-xr-x 2 root root   4096 Mar 23 07:53 LJSpeech_2546vocab_char
drwxr-xr-x 2 root root   4096 Mar 23 07:51 LJSpeech_char
total 2168
drwxr-xr-x 2 root root    4096 Mar 23 07:53 .
drwxr-xr-x 5 root root    4096 Mar 23 07:57 ..
-rw-r--r-- 1 root root  248953 Mar 23 07:53 duration.json
-rw-r--r-- 1 root root 1942064 Mar 23 07:53 raw.arrow
-rw-r--r-- 1 root root   13800 Mar 23 07:53 vocab.txt


In [ ]:
%cd /content/F5-TTS

!python src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_Base \
    --dataset_name LJSpeech_2546vocab \
    --learning_rate 7.5e-5 \
    --batch_size_per_gpu 3800 \
    --max_samples 128 \
    --epochs 1 \
    --save_per_updates 1000 \
    --finetune \
    --pretrain "/content/drive/MyDrive/f5tts_project/models/F5TTS-Base/F5TTS_Base/model_1200000.pt" \
    --tokenizer char \
    --log_samples

/content/F5-TTS

vocab :  2545

vocoder :  vocos
Using logger: None
Loading dataset ...
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
Download Vocos from huggingface charactr/vocos-mel-24khz
config.yaml: 100% 461/461 [00:00<00:00, 2.56MB/s]
pytorch_model.bin: 100% 54.4M/54.4M [00:00<00:00, 65.8MB/s]
Sorting with sampler... if slow, check whether dataset is provided with duration: 100% 13100/131

In [ ]:
# 创建 Drive 保存目录
!mkdir -p /content/drive/MyDrive/f5tts_project/checkpoints

# 复制最终模型
!cp /content/F5-TTS/ckpts/F5TTS_Base/model_last.pt /content/drive/MyDrive/f5tts_project/checkpoints/distilled_model.pt

# 验证
!ls -la /content/drive/MyDrive/f5tts_project/checkpoints/

cp: cannot stat '/content/F5-TTS/ckpts/F5TTS_Base/model_last.pt': No such file or directory
total 0


In [ ]:
# 1. 查看检查点目录
!ls -la /content/F5-TTS/ckpts/F5TTS_Base/

# 2. 如果为空，查看父目录
!ls -la /content/F5-TTS/ckpts/

# 3. 全局搜索 .pt 文件
!find /content/F5-TTS -name "*.pt" -type f 2>/dev/null

ls: cannot access '/content/F5-TTS/ckpts/F5TTS_Base/': No such file or directory
total 16
drwxr-xr-x 3 root root 4096 Mar 23 07:56 .
drwxr-xr-x 7 root root 4096 Mar 23 07:14 ..
drwxr-xr-x 3 root root 4096 Mar 23 09:31 LJSpeech_2546vocab
-rw-r--r-- 1 root root  201 Mar 23 07:13 README.md
/content/F5-TTS/ckpts/LJSpeech_2546vocab/model_2000.pt
/content/F5-TTS/ckpts/LJSpeech_2546vocab/pretrained_model_1200000.pt
/content/F5-TTS/ckpts/LJSpeech_2546vocab/model_1000.pt
/content/F5-TTS/ckpts/LJSpeech_2546vocab/model_last.pt


In [ ]:
# 1. 查看有哪些模型文件
!ls -la /content/F5-TTS/ckpts/LJSpeech_2546vocab/

# 2. 复制最终模型到 Drive
!cp /content/F5-TTS/ckpts/LJSpeech_2546vocab/model_last.pt /content/drive/MyDrive/f5tts_project/checkpoints/distilled_model.pt

# 3. 验证
!ls -la /content/drive/MyDrive/f5tts_project/checkpoints/

total 17120492
drwxr-xr-x 3 root root       4096 Mar 23 09:31 .
drwxr-xr-x 3 root root       4096 Mar 23 07:56 ..
-rw-r--r-- 1 root root 5394193155 Mar 23 08:35 model_1000.pt
-rw-r--r-- 1 root root 5394193155 Mar 23 09:19 model_2000.pt
-rw-r--r-- 1 root root 5394193155 Mar 23 09:39 model_last.pt
-rw------- 1 root root 1348767810 Mar 12 07:22 pretrained_model_1200000.pt
drwxr-xr-x 2 root root       4096 Mar 23 09:19 samples
total 5267767
-rw------- 1 root root 5394193155 Mar 23 09:41 distilled_model.pt


In [ ]:
# 查看 model_last.pt 里有什么
import torch
ckpt = torch.load("/content/F5-TTS/ckpts/LJSpeech_2546vocab/model_last.pt", map_location="cpu")
print(ckpt.keys())  # 看看包含哪些部分

dict_keys(['model_state_dict', 'optimizer_state_dict', 'ema_model_state_dict', 'scheduler_state_dict', 'update'])


In [ ]:
import torch

# 加载检查点
ckpt = torch.load("/content/F5-TTS/ckpts/LJSpeech_2546vocab/model_last.pt", map_location="cpu")

# 提取纯模型权重
model_weights = ckpt["model_state_dict"]

# 保存为纯模型文件
torch.save(model_weights, "/content/F5-TTS/ckpts/LJSpeech_2546vocab/pure_model.pt")

# 验证大小
!ls -lh /content/F5-TTS/ckpts/LJSpeech_2546vocab/pure_model.pt

-rw-r--r-- 1 root root 1.3G Mar 23 09:46 /content/F5-TTS/ckpts/LJSpeech_2546vocab/pure_model.pt


In [ ]:
!cp /content/F5-TTS/ckpts/LJSpeech_2546vocab/pure_model.pt /content/drive/MyDrive/f5tts_project/checkpoints/distilled_model.pt
!ls -lh /content/drive/MyDrive/f5tts_project/checkpoints/distilled_model.pt

-rw------- 1 root root 1.3G Mar 23 09:47 /content/drive/MyDrive/f5tts_project/checkpoints/distilled_model.pt


In [ ]:
# 复制 samples 目录到 Drive
!cp -r /content/F5-TTS/ckpts/LJSpeech_2546vocab/samples /content/drive/MyDrive/f5tts_project/checkpoints/samples

# 验证
!ls -la /content/drive/MyDrive/f5tts_project/checkpoints/samples/

total 1649
-rw------- 1 root root 448078 Mar 23 09:48 update_1000_gen.wav
-rw------- 1 root root 448078 Mar 23 09:48 update_1000_ref.wav
-rw------- 1 root root 395342 Mar 23 09:48 update_2000_gen.wav
-rw------- 1 root root 395342 Mar 23 09:48 update_2000_ref.wav
